In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv("../../data/iris.csv")
df.head()

,Id,Sepal Length (cm),Sepal Width (cm),Petal Length (cm),Petal Width (cm),Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
X = df.drop(["Id", "Species"], axis=1)
y = df["Species"]

y

0         Iris-setosa
1         Iris-setosa
2         Iris-setosa
3         Iris-setosa
4         Iris-setosa
            ...      
145    Iris-virginica
146    Iris-virginica
147    Iris-virginica
148    Iris-virginica
149    Iris-virginica
Name: Species, Length: 150, dtype: object

In [5]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_encode = label_encoder.fit_transform(y)

y_onehot = tf.keras.utils.to_categorical(y_encode)

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_valid, y_train, y_valid = train_test_split(X_scaled, y_onehot, test_size=0.2, random_state=42)

In [14]:
model = keras.Sequential([
    keras.Input([X.shape[1]]),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(3, activation="softmax")
])

In [15]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/50
12/12 - 1s - 54ms/step - accuracy: 0.7604 - loss: 0.7254 - val_accuracy: 0.8750 - val_loss: 0.4993
Epoch 2/50
12/12 - 0s - 6ms/step - accuracy: 0.8646 - loss: 0.3670 - val_accuracy: 0.9167 - val_loss: 0.3722
Epoch 3/50
12/12 - 0s - 5ms/step - accuracy: 0.9271 - loss: 0.2491 - val_accuracy: 0.9167 - val_loss: 0.3046
Epoch 4/50
12/12 - 0s - 5ms/step - accuracy: 0.9375 - loss: 0.1844 - val_accuracy: 0.9583 - val_loss: 0.2567
Epoch 5/50
12/12 - 0s - 5ms/step - accuracy: 0.9479 - loss: 0.1436 - val_accuracy: 0.9583 - val_loss: 0.2478
Epoch 6/50
12/12 - 0s - 5ms/step - accuracy: 0.9688 - loss: 0.1173 - val_accuracy: 0.9583 - val_loss: 0.1929
Epoch 7/50
12/12 - 0s - 5ms/step - accuracy: 0.9479 - loss: 0.1123 - val_accuracy: 0.9583 - val_loss: 0.2541
Epoch 8/50
12/12 - 0s - 5ms/step - accuracy: 0.9583 - loss: 0.0955 - val_accuracy: 0.9583 - val_loss: 0.2053


In [18]:
# Evaluate
test_loss, test_acc = model.evaluate(X_valid, y_valid)
print(f"\nTest Accuracy: {test_acc:.3f}")

# Predict
predictions = model.predict(X_valid)
print(predictions[:5])
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_valid, axis=1)

print("\nPredicted Classes:", predicted_classes[:10])
print("Actual Classes:   ", actual_classes[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.9667 - loss: 0.0939

Test Accuracy: 0.967
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
[[3.1773839e-03 8.8079762e-01 1.1602506e-01]
 [9.9957150e-01 4.2777052e-04 7.1009794e-07]
 [4.1806661e-08 3.2671663e-04 9.9967325e-01]
 [7.0778485e-03 5.7837009e-01 4.1455209e-01]
 [1.3007200e-03 6.0389364e-01 3.9480570e-01]]

Predicted Classes: [1 0 2 1 1 0 1 2 2 1]
Actual Classes:    [1 0 2 1 1 0 1 2 1 1]
